# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [2]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [3]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Use Case Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [4]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/Projects_with_Domains.csv",
    metadata_columns=[
      "Project Title",
      "Project Domain",
      "Secondary Domain",
      "Description",
      "Judge Comments",
      "Score",
      "Project Name",
      "Judge Score"
    ]
)

synthetic_usecase_data = loader.load()

for doc in synthetic_usecase_data:
    doc.page_content = doc.metadata["Description"]

Let's look at an example document to see if everything worked as expected!

In [5]:
synthetic_usecase_data[0]

Document(metadata={'source': './data/Projects_with_Domains.csv', 'row': 0, 'Project Title': 'InsightAI 1', 'Project Domain': 'Security', 'Secondary Domain': 'Finance / FinTech', 'Description': 'A low-latency inference system for multimodal agents in autonomous systems.', 'Judge Comments': 'Technically ambitious and well-executed.', 'Score': '85', 'Project Name': 'Project Aurora', 'Judge Score': '9.5'}, page_content='A low-latency inference system for multimodal agents in autonomous systems.')

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "Synthetic_Usecases".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [6]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    synthetic_usecase_data,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecases"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [7]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [8]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [9]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [10]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [11]:
naive_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain in the provided data is "Healthcare / MedTech," which appears multiple times among the listed projects.'

In [12]:
naive_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are usecases related to security mentioned in the provided context. Specifically, one project named "BioForge" is described as a medical imaging solution improving early diagnosis through vision transformers, and it is categorized under the domain "Security."'

In [13]:
naive_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges\' comments on the fintech projects generally highlight their strengths and areas for improvement. For example, one project was described as having a "comprehensive and technically mature approach," while another was seen as a "clever solution with measurable environmental benefit." Overall, the comments suggest that the fintech-related projects are viewed positively, with attributes such as technical robustness, promising ideas with strong validation, and impressive real-world impact. However, some comments also note that certain projects could benefit from additional benchmarking or qualitative analysis to strengthen their results.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [14]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(synthetic_usecase_data)

We'll construct the same chain - only changing the retriever.

In [15]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [16]:
bm25_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Finance / FinTech," as it is mentioned multiple times in the sample entries.'

In [17]:
bm25_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there was a use case related to security. The project titled "SecureNest 49" falls under the domain of E-commerce / Marketplaces with a secondary domain of Legal / Compliance. Its description indicates that it is a document summarization and retrieval system for enterprise knowledge bases, which suggests a focus on security and compliance in managing sensitive information.'

In [18]:
bm25_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had positive comments about the fintech projects. Specifically, for the project "SynthMind," which is in the Finance / FinTech domain, the judge noted that it has a strong conceptual foundation but mentioned that its results need more benchmarking.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### ✅ Answer

Example Query: “SOC2 encryption requirements”

Why BM25 is better: BM25 focuses on exact keyword matches like “SOC2” and “encryption,” avoiding confusion with loosely related terms.

Real Scenario: If one document says “SOC2 requires encryption at rest” and another says “General data security involves encryption,” BM25 correctly picks the SOC2-specific one, while embeddings might rank the general one higher.


## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [19]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [20]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [21]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain is not explicitly stated as the dominant one. However, among the examples shown, the projects fall into various domains: "Creative / Design / Media," "Security," and "Productivity Assistants." \n\nSince I only have a small sample of projects and no overall frequency count, I cannot definitively identify the most common project domain. If you have a larger dataset or additional information, I can help analyze it to determine the most prevalent domain.'

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no specific use cases related to security. The uses mentioned focus on federated learning to improve privacy in healthcare applications, but there are no explicit references to security use cases.'

In [23]:
contextual_compression_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had positive feedback regarding the fintech projects. Specifically, for the project "WealthifyAI," which is in the finance domain, judges described it as a "comprehensive and technically mature approach." This indicates recognition of its thoroughness and technical quality.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [24]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [25]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [26]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain in the provided data appears to be "Healthcare / MedTech," which is mentioned multiple times.'

In [27]:
multi_query_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are use cases related to security mentioned in the provided context. Specifically, one project titled "SecureNest 28" involves a hardware-aware model quantization benchmark suite with a focus on legal and compliance aspects. The description indicates it is a security-related project, and the judge comments mention that it is conceptually strong.'

In [28]:
multi_query_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges had positive comments about the fintech-related projects. For example, the project "TrendLens" received praise for being "Technically ambitious and well-executed," while "Pathfinder" was described as having a "Promising idea with robust experimental validation." Additionally, "PlanPilot" was noted as "A clever solution with measurable environmental benefit." Overall, judges appreciated the technical strength, impact potential, and innovation of these fintech projects.'

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### ✅ Answer

Generating multiple reformulations of a user query helps improve recall by searching in different ways for the same idea. It’s like having several people describe the same question using different words. This increases the chances of finding documents that use varied terms or phrasing. For example, the query “machine learning algorithms” might also be written as “ML models,” “AI techniques,” or “predictive analytics.” Each version can retrieve different relevant documents, and combining them gives a more complete set of results.


## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [29]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = synthetic_usecase_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [30]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [31]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [32]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [33]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [34]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain is "Healthcare / MedTech," which appears multiple times in the sample.'

In [35]:
parent_document_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no specific use cases related to security explicitly mentioned. The projects primarily focus on federated learning and privacy in healthcare applications, which can be related to security, but there are no direct references to security use cases.'

In [36]:
parent_document_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had positive comments about the fintech projects. For example, they described the project in the finance/fintech domain as "Technically ambitious and well-executed," and another as "A clever solution with measurable environmental benefit." Overall, the judges appreciated the technical maturity and real-world impact of these projects.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [37]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [38]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [39]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Creative / Design / Media," as it is mentioned multiple times in the sample. However, since the data is limited to this excerpt, I cannot definitively confirm the overall most common domain in the entire dataset. \n\nIf you need a precise answer encompassing all entries, I recommend analyzing the full dataset to count the frequency of each domain.'

In [40]:
ensemble_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are usecases related to security. Specifically, there is a project titled "MediMind 17" in the Security domain, which involves a medical imaging solution that aims to improve early diagnosis through vision transformers. Additionally, there is a project called "SecureNest 12" in the Security domain, which focuses on a low-latency inference system for multimodal agents in autonomous systems.'

In [41]:
ensemble_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges\' comments about the fintech projects were generally positive. For example, one project described as "A federated learning toolkit improving privacy in healthcare applications" was praised as having a "Comprehensive and technically mature approach." Another fintech-related project, "An AI model compression suite enabling on-device reasoning for IoT sensors," received an "Excellent code quality and use of open-source libraries." Additionally, a project focused on an "adaptive fine-tuning pipeline for multilingual reasoning models" was noted as "Technically ambitious and well-executed." Overall, judges recognized the innovation, technical strength, and real-world impact of the fintech projects.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [42]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [43]:
semantic_documents = semantic_chunker.split_documents(synthetic_usecase_data[:20])

Let's create a new vector store.

In [44]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecase_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [45]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [46]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [47]:
semantic_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain in the provided data is "Legal / Compliance," which appears twice.'

In [48]:
semantic_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are use cases related to security. Specifically, the projects "SynthMind" with the project "WealthifyAI 3" in the Developer Tools / DevEx domain, and "BioForge" with the project "MediMind 17" in the Security domain, are associated with security applications.'

In [49]:
semantic_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges had the following to say about the fintech projects:\n\n- "Comprehensive and technically mature approach." (Regarding WealthifyAI 16)\n- "Technically ambitious and well-executed." (Regarding TrendLens 19)\n- "Well-structured and scalable; good potential for commercialization." (Regarding WealthifyAI 3)\n- "A forward-looking idea with solid supporting data." (Regarding AutoMate 5)\n\nOverall, judges viewed the fintech projects positively, highlighting their technical maturity, ambition, structure, scalability, and potential for impact.'

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### ✅ Answer
When FAQ sentences are short and repeat a lot, semantic chunking can get confused because all the sentences look very similar. It might split chunks in weird places, even breaking up a question and its answer. To fix this, you can use fixed-size chunks with some overlap or set rules to keep each question and answer together, so the chunks make more sense.


# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [ ]:
# =============================================================================
# STEP 1: LOAD SYNTHETIC DATA AND SETUP
# =============================================================================

import pandas as pd
import os
import time
from uuid import uuid4
from operator import itemgetter

# Load your existing synthetic dataset (s09 assignment data)
dataset = pd.read_csv('test_dataset.csv')
print(f"✅ Loaded {len(dataset)} questions from test_dataset.csv")
print(f"📋 Columns: {list(dataset.columns)}")

# Display sample questions
print("\n📋 Sample Questions:")
for i in range(3):
    print(f"\n{i+1}. {dataset.iloc[i]['user_input']}")
    print(f"   Answer: {dataset.iloc[i]['reference'][:100]}...")
    print(f"   Contexts: {len(dataset.iloc[i]['reference_contexts'])} contexts")

✅ Loaded 4 questions from test_dataset.csv
📋 Columns: ['user_input', 'reference_contexts', 'reference', 'synthesizer_name']

📋 Sample Questions:

1. According to the analysis presented by Handa et al., how does the rapid diffusion and usage of ChatGPT reflect on societal impacts and what are the key considerations highlighted in their study?
   Answer: Handa et al. study consumer usage of ChatGPT, the first mass-market chatbot, and likely the largest,...
   Contexts: 3825 contexts

2. US mean what?
   Answer: The context discusses ChatGPT message usage in the US, including daily message counts, categories of...
   Contexts: 4051 contexts

3. Can you tell me what SOC2 codes 11 means in context of ChatGPT usage?
   Answer: Variation by Occupation Figure 23 presents variation in ChatGPT usage by user occupation, including ...
   Contexts: 3898 contexts


In [115]:
# =============================================================================
# STEP 2: LANGSMITH SETUP WITH GETPASS
# =============================================================================

import getpass

# Enable Langchain tracing (langSmith)
os.environ["LANGCHAIN_TRACING_V2"] = "true"

# Get LangSmith API key using getpass
langsmith_key = getpass.getpass("Enter your LangSmith API Key: ")

# Set the API key
os.environ["LANGCHAIN_API_KEY"] = langsmith_key

# Create langsmith project
os.environ["LANGCHAIN_PROJECT"] = f"AIM - S09-Assignment - {uuid4().hex[0:8]}"

# Create the DS on LangSmith and setting the Client
from langsmith import Client

client = Client()

dataset_name = "Synthetic Data for S09-Assignment"

try:
    langsmith_dataset = client.read_dataset(dataset_name=dataset_name)
    print(f"📂 Using existing LangSmith dataset: {dataset_name}")
except:
    langsmith_dataset = client.create_dataset(
        dataset_name=dataset_name,
        description="SD for Retrievers"
    )
    print(f"📂 Created new LangSmith dataset: {dataset_name}")

# Load questions to LangSmith
for data_row in dataset.iterrows():
    client.create_example(
        inputs={
            "question": data_row[1]["user_input"]
        },
        outputs={
            "answer": data_row[1]["reference"]
        },
        metadata={
            "context": data_row[1]["reference_contexts"]
        },
        dataset_id=langsmith_dataset.id
    )

print("✅ LangSmith setup complete")

📂 Created new LangSmith dataset: Synthetic Data for S09-Assignment
✅ LangSmith setup complete


In [116]:
# =============================================================================
# STEP 3: LOAD PDF DATA AND SETUP COMPONENTS
# =============================================================================

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Qdrant
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.retrievers import ParentDocumentRetriever, EnsembleRetriever
from langchain.storage import InMemoryStore
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_cohere import CohereRerank
from langchain_experimental.text_splitter import SemanticChunker
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from qdrant_client import QdrantClient, models
from langchain_qdrant import QdrantVectorStore

print("📄 Loading PDF data...")

# Load PDF data
loader = PyMuPDFLoader(file_path='data/howpeopleuseai.pdf')
pdf_docs = loader.load()
print(f"✅ Loaded {len(pdf_docs)} documents from PDF")

# Initialize shared components
pdf_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
chat_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Text splitters
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

print("✅ Components initialized")

📄 Loading PDF data...
✅ Loaded 64 documents from PDF
✅ Components initialized


In [117]:
# =============================================================================
# STEP 4: CREATE ALL RETRIEVERS
# =============================================================================

print("🔧 Creating all retrieval strategies...")

# Create vector store
pdf_vectorstore = Qdrant.from_documents(
    documents=pdf_docs,
    embedding=pdf_embeddings,
    location=":memory:",
    collection_name="PDF_Synthetic_Questions"
)

# 1. Naive Retriever
pdf_naive_retriever = pdf_vectorstore.as_retriever(search_kwargs={"k": 10})

# 2. BM25 Retriever
pdf_bm25_retriever = BM25Retriever.from_documents(pdf_docs)

# 3. Compression Retriever
compressor = CohereRerank(model="rerank-v3.5")
pdf_compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, 
    base_retriever=pdf_naive_retriever
)

# 4. Multi-Query Retriever
pdf_multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=pdf_naive_retriever, 
    llm=chat_model
)

# 5. Parent Document Retriever
pdf_parent_docs = pdf_docs

pdf_parent_client = QdrantClient(location=":memory:")

pdf_parent_client.create_collection(
    collection_name="pdf_full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

pdf_parent_document_vectorstore = QdrantVectorStore(
    collection_name="pdf_full_documents", 
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"), 
    client=pdf_parent_client
)

pdf_parent_store = InMemoryStore()

pdf_parent_retriever = ParentDocumentRetriever(
    vectorstore=pdf_parent_document_vectorstore,
    docstore=pdf_parent_store,
    child_splitter=child_splitter,
)

pdf_parent_retriever.add_documents(pdf_parent_docs, ids=None)

# 6. Ensemble Retriever
pdf_retriever_list = [pdf_naive_retriever, pdf_bm25_retriever, pdf_compression_retriever, pdf_multi_query_retriever, pdf_parent_retriever]
equal_weighting = [1/len(pdf_retriever_list)] * len(pdf_retriever_list)

pdf_ensemble_retriever = EnsembleRetriever(retrievers=pdf_retriever_list, weights=equal_weighting)

# 7. Semantic Chunking
semantic_chunker = SemanticChunker(
    pdf_embeddings,
    breakpoint_threshold_type="percentile"
)

pdf_semantic_documents = semantic_chunker.split_documents(pdf_docs)

pdf_semantic_vectorstore = Qdrant.from_documents(
    pdf_semantic_documents,
    pdf_embeddings,
    location=":memory:",
    collection_name="Synthetic_PDF_Data_Semantic_Chunks"
)

pdf_semantic_retriever = pdf_semantic_vectorstore.as_retriever(search_kwargs={"k": 10})

print("✅ All retrievers created")

🔧 Creating all retrieval strategies...
✅ All retrievers created


In [118]:
# =============================================================================
# STEP 5: CREATE RAG TEMPLATE AND CHAINS
# =============================================================================

print("🔗 Creating RAG template and chains...")

# RAG template
RAG_TEMPLATE = """You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}"""

# Create prompt template
rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

# LCEL RAG Chain Function
def lcel_chain(retriever):
    return (
        {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
        | RunnablePassthrough.assign(context=itemgetter("context"))
        | {"output": rag_prompt | chat_model, "context": itemgetter("context")}
    )

# Create chains for all retrievers
pdf_naive_retriever_chain = lcel_chain(pdf_naive_retriever)
pdf_bm25_retriever_chain = lcel_chain(pdf_bm25_retriever)
pdf_compression_retriever_chain = lcel_chain(pdf_compression_retriever)
pdf_multi_query_retriever_chain = lcel_chain(pdf_multi_query_retriever)
pdf_parent_retriever_chain = lcel_chain(pdf_parent_retriever)
pdf_ensemble_retriever_chain = lcel_chain(pdf_ensemble_retriever)
pdf_semantic_retriever_chain = lcel_chain(pdf_semantic_retriever)

print("✅ RAG chains created for all retrievers")

🔗 Creating RAG template and chains...
✅ RAG chains created for all retrievers


In [119]:
# =============================================================================
# STEP 6: EVALUATION SETUP
# =============================================================================

from langsmith.evaluation import evaluate as langsmith_evaluate

# Storage for RAGAS
captured_results = {}

# Wrapper that captures results while LangSmith runs
class CapturingChain:
    def __init__(self, chain, storage_key, dataset_df, delay_seconds=0):
        self.chain = chain
        self.storage_key = storage_key
        self.dataset_df = dataset_df
        self.delay_seconds = delay_seconds
        self.call_count = 0
        captured_results[storage_key] = []
    
    def invoke(self, inputs):
        if self.call_count > 0 and self.delay_seconds > 0:
            time.sleep(self.delay_seconds)
        
        # Run the chain
        output = self.chain.invoke(inputs)
        
        # Capture for RAGAS
        question = inputs["question"]
        matching_row = self.dataset_df[self.dataset_df["user_input"] == question]
        if not matching_row.empty:
            captured_results[self.storage_key].append({
                "user_input": question,
                "response": output["output"].content,
                "retrieved_contexts": [doc.page_content for doc in output["context"]],
                "reference": matching_row.iloc[0]["reference"],
                "reference_contexts": matching_row.iloc[0]["reference_contexts"]
            })
        
        self.call_count += 1
        return output

print("✅ Evaluation setup complete")

✅ Evaluation setup complete


In [120]:
# =============================================================================
# STEP 7: RAGAS EVALUATION SETUP
# =============================================================================

from ragas import EvaluationDataset
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import (
    Faithfulness, 
    FactualCorrectness, 
    ResponseRelevancy, 
    ContextEntityRecall,
    ContextPrecision,
    ContextRecall,
)
from ragas import evaluate as ragas_evaluate
from ragas import RunConfig

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))
custom_run_config = RunConfig(
    timeout=600,
    max_workers=4
)

# Evaluate function
def evaluate_with_ragas(results, retriever_name):
    """Evaluate a retriever's results with RAGAS metrics"""
    print(f"Evaluating {retriever_name}...")
    
    # Convert to RAGAS dataset
    ragas_dataset = EvaluationDataset.from_list(results)
    
    # Evaluate with BOTH generation AND retriever metrics
    scores = ragas_evaluate(
        dataset=ragas_dataset,
        metrics=[
            # Retriever-specific metrics (main focus for assignment)
            ContextPrecision(),       # How precise is retrieval?
            ContextRecall(),          # How complete is retrieval?
            ContextEntityRecall(),    # Are key entities retrieved?
            # Generation metrics (show overall quality)
            Faithfulness(),           # Is response faithful to context?
            FactualCorrectness(),     # Is response factually correct?
            ResponseRelevancy(),      # Is response relevant to question?
        ],
        llm=evaluator_llm,
        run_config=custom_run_config
    )
    
    print(f"{retriever_name} completed!")
    return scores

print("✅ RAGAS evaluation setup complete")

✅ RAGAS evaluation setup complete


C:\Users\Chandu\AppData\Local\Temp\ipykernel_15900\11285694.py:18: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use the modern LLM providers instead: from ragas.llms.base import llm_factory; llm = llm_factory('gpt-4o-mini') or from ragas.llms.base import instructor_llm_factory; llm = instructor_llm_factory('openai', client=openai_client)
  evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))


In [121]:
# =============================================================================
# STEP 8: RUN LANGSMITH EVALUATIONS
# =============================================================================

print("🚀 Running LangSmith evaluations...")

# Run LangSmith evaluations (chains run once, results captured)
configs = [
    (pdf_naive_retriever_chain, "naive-retriever", "Naive Retriever", 0),
    (pdf_bm25_retriever_chain, "bm25-retriever", "BM25 Retriever", 0),
    (pdf_compression_retriever_chain, "compression-retriever", "Compression Retriever", 7),
    (pdf_multi_query_retriever_chain, "multiquery-retriever", "Multi-Query Retriever", 0),
    (pdf_parent_retriever_chain, "parent-retriever", "Parent Retriever", 0),
    (pdf_ensemble_retriever_chain, "ensemble-retriever", "Ensemble Retriever", 7),
    (pdf_semantic_retriever_chain, "semantic-retriever", "Semantic Chunk Retriever", 0),
]

langsmith_results = {}

for chain, exp_name, display_name, delay in configs:
    print(f"Running {display_name}...")
    capturing_chain = CapturingChain(chain, display_name, dataset, delay)
    langsmith_results[exp_name] = langsmith_evaluate(
        capturing_chain.invoke,
        data="Synthetic Data for S09-Assignment",
        experiment_prefix=exp_name,
    )
    print(f"{display_name} completed!\n")

print("✅ All LangSmith evaluations completed")

🚀 Running LangSmith evaluations...
Running Naive Retriever...
View the evaluation results for experiment: 'naive-retriever-f1b37dcb' at:
https://smith.langchain.com/o/6b208d6f-f67e-4438-9319-6878a8bf7ac7/datasets/da6c829f-facf-4b1d-8304-5b054ceb5938/compare?selectedSessions=17f16fe5-4b69-45fb-b0d6-4466ec05aab6




4it [00:25,  6.46s/it]


Naive Retriever completed!

Running BM25 Retriever...
View the evaluation results for experiment: 'bm25-retriever-1a6895d0' at:
https://smith.langchain.com/o/6b208d6f-f67e-4438-9319-6878a8bf7ac7/datasets/da6c829f-facf-4b1d-8304-5b054ceb5938/compare?selectedSessions=6cdc88ef-543e-4776-a136-0c3c36102574




4it [00:17,  4.40s/it]


BM25 Retriever completed!

Running Compression Retriever...
View the evaluation results for experiment: 'compression-retriever-25f491e3' at:
https://smith.langchain.com/o/6b208d6f-f67e-4438-9319-6878a8bf7ac7/datasets/da6c829f-facf-4b1d-8304-5b054ceb5938/compare?selectedSessions=c049c3f1-e55f-4685-9967-399d8b620f9f




...

Parent Retriever completed!

Running Ensemble Retriever...
View the evaluation results for experiment: 'ensemble-retriever-d96d57b8' at:
https://smith.langchain.com/o/6b208d6f-f67e-4438-9319-6878a8bf7ac7/datasets/da6c829f-facf-4b1d-8304-5b054ceb5938/compare?selectedSessions=f8e16e06-bbd7-456b-b6df-deb3d15d26d9




4it [01:13, 18.28s/it]


Ensemble Retriever completed!

Running Semantic Chunk Retriever...
View the evaluation results for experiment: 'semantic-retriever-6718199b' at:
https://smith.langchain.com/o/6b208d6f-f67e-4438-9319-6878a8bf7ac7/datasets/da6c829f-facf-4b1d-8304-5b054ceb5938/compare?selectedSessions=428c5a40-6ec6-4df0-affc-daf6b0fc23f8




4it [00:21,  5.41s/it]

Semantic Chunk Retriever completed!

✅ All LangSmith evaluations completed


In [124]:
# =============================================================================
# STEP 9: RUN RAGAS ANALYSIS - CORRECTED COLUMN NAMES
# =============================================================================

import ast
import json

print("🔍 Running RAGAS analysis...")

# Fixed evaluate function with correct column names
def evaluate_with_ragas_corrected(results, retriever_name):
    """Evaluate a retriever's results with RAGAS metrics - corrected column names"""
    print(f"Evaluating {retriever_name}...")
    
    try:
        # Convert to RAGAS dataset format with correct column names
        ragas_data = []
        for result in results:
            # Fix reference_contexts - convert from string to list
            reference_contexts = result["reference_contexts"]
            if isinstance(reference_contexts, str):
                try:
                    # Try to parse as Python literal first
                    reference_contexts = ast.literal_eval(reference_contexts)
                except:
                    try:
                        # Try JSON parsing
                        reference_contexts = json.loads(reference_contexts)
                    except:
                        # If all else fails, create a single-item list
                        reference_contexts = [reference_contexts]
            
            # Ensure it's a list
            if not isinstance(reference_contexts, list):
                reference_contexts = [reference_contexts]
            
            # Fix retrieved_contexts - ensure it's a list
            retrieved_contexts = result["retrieved_contexts"]
            if not isinstance(retrieved_contexts, list):
                retrieved_contexts = [retrieved_contexts]
            
            # Create the data entry with CORRECT column names for RAGAS
            ragas_entry = {
                "user_input": result["user_input"],           # Correct column name
                "reference": result["reference"],             # Correct column name  
                "retrieved_contexts": retrieved_contexts,     # Correct column name
                "response": result["response"]                # Additional column for generation metrics
            }
            
            ragas_data.append(ragas_entry)
        
        print(f"📊 Sample data for {retriever_name}:")
        print(f"  Question: {ragas_data[0]['user_input'][:50]}...")
        print(f"  Retrieved contexts count: {len(ragas_data[0]['retrieved_contexts'])}")
        print(f"  Reference: {ragas_data[0]['reference'][:50]}...")
        print(f"  Response: {ragas_data[0]['response'][:50]}...")
        
        # Create RAGAS dataset
        ragas_dataset = EvaluationDataset.from_list(ragas_data)
        
        # Evaluate with retriever-specific metrics
        scores = ragas_evaluate(
            dataset=ragas_dataset,
            metrics=[
                ContextPrecision(),       # How precise is retrieval?
                ContextRecall(),          # How complete is retrieval?
                ContextEntityRecall(),    # Are key entities retrieved?
                Faithfulness(),           # Is response faithful to context?
                FactualCorrectness(),     # Is response factually correct?
                ResponseRelevancy(),      # Is response relevant to question?
            ],
            llm=evaluator_llm,
            run_config=custom_run_config
        )
        
        print(f"✅ {retriever_name} completed!")
        return scores
        
    except Exception as e:
        print(f"❌ Error evaluating {retriever_name}: {e}")
        print(f"Debug info:")
        print(f"  Results count: {len(results)}")
        if results:
            print(f"  Sample result keys: {list(results[0].keys())}")
        return None

# Run RAGAS with captured results
ragas_scores = {}
for display_name, results in captured_results.items():
    if results and len(results) > 0:
        print(f"\n🔍 Processing {display_name} with {len(results)} results...")
        try:
            ragas_scores[display_name] = evaluate_with_ragas_corrected(results, display_name)
        except Exception as e:
            print(f"❌ Failed to evaluate {display_name}: {e}")
            # Try with minimal metrics
            try:
                print(f"🔄 Trying with minimal metrics for {display_name}...")
                
                # Create minimal dataset with correct column names
                simple_data = []
                for result in results:
                    # Force convert everything to proper types with correct column names
                    simple_data.append({
                        "user_input": str(result["user_input"]),
                        "reference": str(result["reference"]),
                        "retrieved_contexts": [str(ctx) for ctx in result["retrieved_contexts"]] if isinstance(result["retrieved_contexts"], list) else [str(result["retrieved_contexts"])],
                        "response": str(result["response"])
                    })
                
                ragas_dataset = EvaluationDataset.from_list(simple_data)
                scores = ragas_evaluate(
                    dataset=ragas_dataset,
                    metrics=[ContextPrecision(), Faithfulness()],  # Just 2 metrics
                    llm=evaluator_llm,
                    run_config=custom_run_config
                )
                ragas_scores[display_name] = scores
                print(f"✅ {display_name} completed with minimal metrics")
                
            except Exception as e2:
                print(f"❌ Even minimal evaluation failed for {display_name}: {e2}")
    else:
        print(f"⚠️ No results captured for {display_name}")

print("✅ RAGAS analysis completed")

🔍 Running RAGAS analysis...

🔍 Processing Naive Retriever with 4 results...
Evaluating Naive Retriever...
📊 Sample data for Naive Retriever:
  Question: How does Writing relate to ChatGPT usage?...
  Retrieved contexts count: 10
  Reference: Writing is by far the most common work use, accoun...
  Response: Writing is a significant aspect of ChatGPT usage, ...


Evaluating: 100%|██████████| 24/24 [05:26<00:00, 13.61s/it]


✅ Naive Retriever completed!

🔍 Processing BM25 Retriever with 4 results...
Evaluating BM25 Retriever...
📊 Sample data for BM25 Retriever:
  Question: How does Writing relate to ChatGPT usage?...
  Retrieved contexts count: 4
  Reference: Writing is by far the most common work use, accoun...
  Response: Writing is a significant aspect of ChatGPT usage, ...


Evaluating: 100%|██████████| 24/24 [04:19<00:00, 10.80s/it]


✅ BM25 Retriever completed!

🔍 Processing Compression Retriever with 4 results...
Evaluating Compression Retriever...
📊 Sample data for Compression Retriever:
  Question: How does Writing relate to ChatGPT usage?...
  Retrieved contexts count: 3
  Reference: Writing is by far the most common work use, accoun...
  Response: Writing is a significant aspect of ChatGPT usage, ...


...

✅ Parent Retriever completed!

🔍 Processing Ensemble Retriever with 4 results...
Evaluating Ensemble Retriever...
📊 Sample data for Ensemble Retriever:
  Question: How does Writing relate to ChatGPT usage?...
  Retrieved contexts count: 15
  Reference: Writing is by far the most common work use, accoun...
  Response: Writing is a significant aspect of ChatGPT usage, ...


Evaluating: 100%|██████████| 24/24 [07:01<00:00, 17.56s/it]


✅ Ensemble Retriever completed!

🔍 Processing Semantic Chunk Retriever with 4 results...
Evaluating Semantic Chunk Retriever...
📊 Sample data for Semantic Chunk Retriever:
  Question: How does Writing relate to ChatGPT usage?...
  Retrieved contexts count: 10
  Reference: Writing is by far the most common work use, accoun...
  Response: Writing is a significant aspect of ChatGPT usage, ...


Evaluating: 100%|██████████| 24/24 [05:04<00:00, 12.68s/it]


✅ Semantic Chunk Retriever completed!
✅ RAGAS analysis completed


In [125]:
# =============================================================================
# STEP 10: GENERATE RESULTS TABLES
# =============================================================================

print("📊 Generating results tables...")

# TABLE 1: Aggregated Average Scores per Retriever (TRANSPOSED)
all_results = []
for name in ["Naive Retriever", "BM25 Retriever", "Compression Retriever", 
             "Multi-Query Retriever", "Parent Retriever", "Ensemble Retriever", 
             "Semantic Chunk Retriever"]:
    if name in ragas_scores:
        df = ragas_scores[name].to_pandas()
        df.insert(0, 'retriever', name)
        all_results.append(df)

if all_results:
    results_comparison = pd.concat(all_results, ignore_index=True)
    
    # Get metric columns (exclude metadata)
    metric_columns = [col for col in results_comparison.columns 
                      if col not in ['retriever', 'user_input', 'retrieved_contexts', 
                                     'reference', 'reference_contexts', 'response']]
    
    # Calculate mean scores per retriever
    aggregated_scores = results_comparison.groupby('retriever')[metric_columns].mean()
    
    # Reorder rows
    aggregated_scores = aggregated_scores.reindex(["Naive Retriever", "BM25 Retriever", 
                                                    "Compression Retriever", "Multi-Query Retriever", 
                                                    "Parent Retriever", "Ensemble Retriever", 
                                                    "Semantic Chunk Retriever"])
    
    # TRANSPOSE: metrics as rows, retrievers as columns
    aggregated_scores = aggregated_scores.T
    
    print("📊 TABLE 1: Average Scores per Retriever")
    print("=" * 120)
    with pd.option_context('display.max_columns', None, 
                           'display.width', None,
                           'display.float_format', '{:.4f}'.format):
        display(aggregated_scores)
else:
    print("⚠️ No RAGAS results available to display")

print("✅ Results tables generated")

📊 Generating results tables...
📊 TABLE 1: Average Scores per Retriever


retriever,Naive Retriever,BM25 Retriever,Compression Retriever,Multi-Query Retriever,Parent Retriever,Ensemble Retriever,Semantic Chunk Retriever
context_precision,0.7407,0.6458,0.7500,0.7131,0.7500,0.7104,0.6394
context_recall,1.0000,0.8750,0.7917,1.0000,0.8750,1.0000,1.0000
context_entity_recall,0.1080,0.1080,0.1750,0.1267,0.2640,0.1807,0.1958
faithfulness,0.8699,0.8991,0.8065,0.6979,0.9444,0.9821,0.8681
factual_correctness(mode=f1),0.2750,0.4200,0.3175,0.3700,0.3675,0.4150,0.3225
answer_relevancy,0.9309,0.7236,0.7201,0.7206,0.9167,0.7224,0.7061


✅ Results tables generated


In [147]:
# =============================================================================
# STEP 13: FINAL SUMMARY AND RECOMMENDATIONS
# =============================================================================

print("🏆 FINAL SUMMARY AND RECOMMENDATIONS")
print("=" * 80)

if all_results:
    # Calculate overall performance
    performance_summary = aggregated_scores.mean(axis=1).sort_values(ascending=False)
    
    print("\n📊 OVERALL PERFORMANCE RANKING:")
    print("-" * 40)
    for i, (metric, score) in enumerate(performance_summary.items(), 1):
        print(f"{i:2d}. {metric}: {score:.4f}")
    
    # Find best retriever for each metric
    print("\n🏆 BEST RETRIEVER BY METRIC:")
    print("-" * 40)
    for metric in metric_columns:
        best_retriever = aggregated_scores.loc[metric].idxmax()
        best_score = aggregated_scores.loc[metric].max()
        print(f"{metric}: {best_retriever} ({best_score:.4f})")
    
    # Overall best retriever
    retriever_avg_scores = aggregated_scores.mean(axis=0).sort_values(ascending=False)
    best_overall = retriever_avg_scores.index[0]
    best_score = retriever_avg_scores.iloc[0]
    
    print(f"\n🥇 OVERALL BEST RETRIEVER: {best_overall}")
    print(f"📈 Average Score: {best_score:.4f}")
    
    

print(f"\n✅ Activity 1 - Advanced Retrieval Evaluation Completed!")
print(f"📁 Results saved and analysis complete")
print(f"🔗 Check LangSmith dashboard for detailed traces: https://smith.langchain.com/")

🏆 FINAL SUMMARY AND RECOMMENDATIONS

📊 OVERALL PERFORMANCE RANKING:
----------------------------------------
 1. context_recall: 0.9345
 2. faithfulness: 0.8669
 3. answer_relevancy: 0.7772
 4. context_precision: 0.7070
 5. factual_correctness(mode=f1): 0.3554
 6. context_entity_recall: 0.1654

🏆 BEST RETRIEVER BY METRIC:
----------------------------------------
context_precision: Compression Retriever (0.7500)
context_recall: Naive Retriever (1.0000)
context_entity_recall: Parent Retriever (0.2640)
faithfulness: Ensemble Retriever (0.9821)
factual_correctness(mode=f1): BM25 Retriever (0.4200)
answer_relevancy: Naive Retriever (0.9309)

🥇 OVERALL BEST RETRIEVER: Parent Retriever
📈 Average Score: 0.6863

✅ Activity 1 - Advanced Retrieval Evaluation Completed!
📁 Results saved and analysis complete
🔗 Check LangSmith dashboard for detailed traces: https://smith.langchain.com/


In [146]:
# =============================================================================
# STEP 14: SAVE FINAL RESULTS
# =============================================================================

print("💾 Saving final results...")

# Save aggregated scores
if all_results:
    aggregated_scores.to_csv('retriever_evaluation_results.csv')
    print("✅ Results saved to 'retriever_evaluation_results.csv'")

# Save detailed comparison
if all_results:
    results_comparison.to_csv('detailed_retriever_comparison.csv', index=False)
    print("✅ Detailed comparison saved to 'detailed_retriever_comparison.csv'")


print("🎉 All results saved successfully!")
print("📊 Activity 1 Complete - Advanced Retrieval Evaluation")

💾 Saving final results...
✅ Results saved to 'retriever_evaluation_results.csv'
✅ Detailed comparison saved to 'detailed_retriever_comparison.csv'
🎉 All results saved successfully!
📊 Activity 1 Complete - Advanced Retrieval Evaluation


#### Best Choice for This Dataset:

The Parent Retriever emerges as the optimal choice, achieving the highest overall quality score (0.6863) with the fastest response time (3.506s) and lowest cost ($0.0021). While specialized retrievers excel in specific areas—Naive Retriever's perfect context recall (1.0000) and Ensemble Retriever's superior faithfulness (0.9821)—the Parent Retriever provides the best balanced performance across all metrics. The system shows strong performance in context recall (0.9345) and faithfulness (0.8669), though factual correctness (0.3554) and entity recall (0.1654) need improvement. For production environments, the Parent Retriever offers the ideal combination of quality, speed, and cost-effectiveness, making it the clear winner for enterprise RAG applications.

#### Cost & Latency Analysis by Experiment(langsmith)

##### Experiment Performance Summary

| Experiment | Retriever Type | Avg Latency (s) | Avg Cost ($) | Total Tokens | Performance Grade |
|---|---|---|---|---|---|
| `parent-retriever-f724bb76` | Parent Retriever | **3.506** | **$0.0021** | 11,988 | 🥇 **A+** |
| `semantic-retriever-6718199b` | Semantic Chunk Retriever | 3.639 | $0.0033 | 19,751 | 🥈 **A** |
| `bm25-retriever-1a6895d0` | BM25 Retriever | 4.015 | $0.0023 | 13,747 | 🥉 **B+** |
| `naive-retriever-f1b37dcb` | Naive Retriever | 4.514 | $0.0048 | 29,709 | **C+** |
| `multiquery-retriever-0f8d2d27` | Multi-Query Retriever | 7.127 | $0.0069 | 44,659 | **D+** |
| `compression-retriever-25f491e3` | Compression Retriever | 10.460 | $0.0021 | 11,664 | **D** |
| `ensemble-retriever-d96d57b8` | Ensemble Retriever | 15.320 | $0.0091 | 58,570 | **F** |

##### Individual Run Details

**Parent Retriever (`parent-retriever-f724bb76`)**
| Query | Latency (s) | Cost ($) |
|---|---|---|
| Handa et al. analysis | 10.18 | $0.001 |
| Writing & ChatGPT | 4.46 | $0.00 |
| SOC2 codes | 2.55 | $0.001 |
| General query | 1.83 | $0.001 |

**Semantic Chunk Retriever (`semantic-retriever-6718199b`)**
| Query | Latency (s) | Cost ($) |
|---|---|---|
| Handa et al. | 10.97 | $0.001 |
| Writing | 4.62 | $0.001 |
| SOC2 codes | 2.66 | $0.001 |
| General | 2.39 | $0.001 |

**BM25 Retriever (`bm25-retriever-1a6895d0`)**
| Query | Latency (s) | Cost ($) |
|---|---|---|
| Handa et al. | 6.57 | $0.001 |
| Writing | 4.89 | $0.001 |
| SOC2 codes | 3.14 | $0.00 |
| General | 2.28 | $0.00 |

**Naive Retriever (`naive-retriever-f1b37dcb`)**
| Query | Latency (s) | Cost ($) |
|---|---|---|
| Handa et al. | 14.06 | $0.002 |
| Writing | 5.95 | $0.001 |
| SOC2 codes | 3.08 | $0.001 |
| General | 2.04 | $0.001 |

**Multi-Query Retriever (`multiquery-retriever-0f8d2d27`)**
| Query | Latency (s) | Cost ($) |
|---|---|---|
| Handa et al. | 15.02 | $0.002 |
| Writing | 8.62 | $0.002 |
| SOC2 codes | 5.63 | $0.002 |
| General | 3.62 | $0.002 |

**Compression Retriever (`compression-retriever-25f491e3`)**
| Query | Latency (s) | Cost ($) |
|---|---|---|
| Handa et al. | 17.90 | $0.001 |
| Writing | 6.99 | $0.001 |
| SOC2 codes | 11.10 | $0.00 |
| General | 9.82 | $0.00 |

**Ensemble Retriever (`ensemble-retriever-d96d57b8`)**
| Query | Latency (s) | Cost ($) |
|---|---|---|
| Handa et al. | 31.69 | $0.002 |
| Writing | 9.85 | $0.002 |
| SOC2 codes | 14.92 | $0.002 |
| General | 15.73 | $0.003 |

##### Performance Rankings

**Fastest Latency**
1. **Parent Retriever**: 3.506s
2. **Semantic Chunk**: 3.639s  
3. **BM25**: 4.015s
4. **Naive**: 4.514s

**Most Cost-Effective**
1. **Parent Retriever**: $0.0021
2. **Compression Retriever**: $0.0021
3. **BM25**: $0.0023
4. **Semantic Chunk**: $0.0033

**Recommendation**: Parent Retriever offers the best balance of speed and cost efficiency. 